In [ ]:
!wget https://files.rcsb.org/pub/pdb/derived_data/pdb_seqres.txt.gz
!gunzip pdb_seqres.txt.gz

--2026-04-16 22:47:56--  https://files.rcsb.org/pub/pdb/derived_data/pdb_seqres.txt.gz
Resolving files.rcsb.org (files.rcsb.org)... 18.64.236.48, 18.64.236.125, 18.64.236.33, ...
Connecting to files.rcsb.org (files.rcsb.org)|18.64.236.48|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63777928 (61M) [application/gzip]
Saving to: ‘pdb_seqres.txt.gz’

pdb_seqres.txt.gz   100%[===================>]  60.82M  31.6MB/s    in 1.9s    

2026-04-16 22:47:58 (31.6 MB/s) - ‘pdb_seqres.txt.gz’ saved [63777928/63777928]



In [ ]:
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.5 MB/s eta 0:00:00


Pull inicial

In [ ]:
from Bio import SeqIO
import pandas as pd

def parse_pdb_seqres(file_path):
    data = []
    for record in SeqIO.parse(file_path, "fasta"):
        # Extraer ID del PDB y longitud
        pdb_id = record.id.split('_')[0]
        sequence = str(record.seq)

        if 50 <= len(sequence) <= 500 and all(c in "ACDEFGHIKLMNPQRSTVWY" for c in sequence):
            data.append({"pdb_id": pdb_id, "sequence": sequence, "length": len(sequence)})

    return pd.DataFrame(data).drop_duplicates(subset='sequence')

df_all = parse_pdb_seqres("pdb_seqres.txt")
print(f"Proteínas candidatas encontradas: {len(df_all)}")

Proteínas candidatas encontradas: 143338


In [ ]:
import requests
import pandas as pd
import time

def obtener_candidatos_recientes(limite=10000):
    url_search = "https://search.rcsb.org/rcsbsearch/v2/query"

    query = {
      "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
          {
            "type": "terminal",
            "service": "text",
            "parameters": {
              "attribute": "rcsb_accession_info.initial_release_date",
              "operator": "greater",
              "value": "2020-01-01T00:00:00Z" #Fecha de corte de entrenamiento
            }
          },
          {
            "type": "terminal",
            "service": "text",
            "parameters": {
              "attribute": "rcsb_entry_info.polymer_entity_count_protein",
              "operator": "equals",
              "value": 1
            }
          }
        ]
      },
      "return_type": "polymer_entity",
      "request_options": {
        "paginate": {"start": 0, "rows": limite}
      }
    }

    response = requests.post(url_search, json=query)
    if response.status_code != 200:
        print("Error en la API:", response.text)
        return []

    resultados = response.json().get('result_set', [])
    ids_encontrados = [item['identifier'] for item in resultados]
    print(f"Se encontraron {len(ids_encontrados)} candidatas.")
    return ids_encontrados

def obtener_secuencias(entity_ids):
    print("Descargando las secuencias")
    datos = []
    for ent_id in entity_ids:
        pdb_id, entity_num = ent_id.split('_')
        url_data = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_num}"

        try:
            res = requests.get(url_data)
            if res.status_code == 200:
                data = res.json()
                seq = data['entity_poly']['pdbx_seq_one_letter_code_can']
                # Filtro de longitud (50 a 300 aa)
                if 50 <= len(seq) <= 300:
                    datos.append({'PDB_ID': pdb_id, 'Secuencia': seq, 'Longitud': len(seq)})
        except Exception as e:
            continue

        time.sleep(0.1)

    df = pd.DataFrame(datos).drop_duplicates(subset='Secuencia')
    print(f"Secuencias finales válidas descargadas: {len(df)}")
    return df

candidatos_ids = obtener_candidatos_recientes(10000) # Pedimos 10000 para tener de sobra
if candidatos_ids:
    df_secuencias = obtener_secuencias(candidatos_ids)
    display(df_secuencias.head())

Se encontraron 10000 candidatas.
Descargando las secuencias
Secuencias finales válidas descargadas: 2084


,PDB_ID,Secuencia,Longitud
0,10AF,MAHHHHHHMSRPHVFFDITIGGSNAGRIVMELFADIVPKTAENFRC...,179
1,10DC,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...,166
2,10DJ,VWEIPRESLQLIKRLGNGQFGEVWMGTWNGNTKVAIKTLKPGTMSP...,282
3,10HY,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,297
6,10IJ,GKVQIINKKLDLSNVQSKCGSKDNIKHVPGGGIVQIVYKPVDLSKV...,107


In [ ]:
import requests
import pandas as pd
import time

def clasificar_secuencias(df):
    novel = []
    similar = []

    url_search = "https://search.rcsb.org/rcsbsearch/v2/query"

    print(f"Clasificando {len(df)} secuencias... Esto tomará unos minutos.")

    for index, row in df.iterrows():
        secuencia = row['Secuencia']
        pdb_id = row['PDB_ID']

        # Construimos la consulta para la API
        query = {
          "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
              {
                "type": "terminal",
                "service": "sequence",
                "parameters": {
                  "evalue_cutoff": 1.0,
                  "identity_cutoff": 0.3, # Umbral del 30% del paper
                  "sequence_type": "protein",
                  "value": secuencia
                }
              },
              {
                "type": "terminal",
                "service": "text",
                "parameters": {
                  "attribute": "rcsb_accession_info.initial_release_date",
                  "operator": "less_or_equal",
                  "value": "2020-01-01T00:00:00Z" # Fecha de corte del paper
                }
              }
            ]
          },
          "return_type": "polymer_entity",
          "request_options": {
            "paginate": {"start": 0, "rows": 1} # Solo necesitamos saber si hay al menos 1
          }
        }

        try:
            response = requests.post(url_search, json=query)

            # 200 significa que encontró resultados (Similar)
            if response.status_code == 200:
                data = response.json()
                if 'result_set' in data and len(data['result_set']) > 0:
                    similar.append(row)
                else:
                    novel.append(row)
            # 204 significa "No Content", es decir, 0 coincidencias (Novel)
            elif response.status_code == 204:
                novel.append(row)
            else:
                print(f"Advertencia con {pdb_id}: Status {response.status_code}")

        except Exception as e:
            print(f"Error de conexión con {pdb_id}: {e}")

        # Pausa de 0.5s para no saturar los servidores del PDB
        time.sleep(0.5)

    df_novel = pd.DataFrame(novel)
    df_similar = pd.DataFrame(similar)

    print(f"\n--- Clasificación Terminada ---")
    print(f"Proteínas 'Novel' (<30% homología pre-2020): {len(df_novel)}")
    print(f"Proteínas 'Similar' (>=30% homología pre-2020): {len(df_similar)}")

    return df_novel, df_similar

df_novel, df_similar = clasificar_secuencias(df_secuencias)

# Mostramos cuántas logramos rescatar
print("Muestra de Novel:")
display(df_novel.head())

Clasificando 2084 secuencias... Esto tomará unos minutos.
Advertencia con 21CB: Status 500
Advertencia con 21CQ: Status 500
Advertencia con 6LTP: Status 500
Advertencia con 6LTU: Status 500
Advertencia con 6LU0: Status 500
Advertencia con 6LVR: Status 500
Advertencia con 6M0V: Status 500
Advertencia con 6M0W: Status 500
Advertencia con 6O8Q: Status 500
Advertencia con 6O8Q: Status 500
Advertencia con 6R7G: Status 500
Advertencia con 6VRC: Status 500
Advertencia con 6W5C: Status 500
Advertencia con 6WW6: Status 500

--- Clasificación Terminada ---
Proteínas 'Novel' (<30% homología pre-2021): 223
Proteínas 'Similar' (>=30% homología pre-2021): 1847
Muestra de Novel:


,PDB_ID,Secuencia,Longitud
6,10JX,GEFLELMRQENAQLISQLRNAVIQDPDENSFYYDLIDNAPDAMVLV...,145
81,22OC,DPYLSKPHLTTTTTGLGQDTPYNCAPNSLQQALYKLLRVKISEKTL...,193
108,28TN,MNNLSKIRRRAGLTQRQIAKELNLTAGAICHYENGKRALSIEQCRK...,82
118,5BKA,MTITALPTGLYAEVLSFYGHQMQKLDGRDFAGYAATFTEDGEFRHS...,146
1078,6HCZ,MALNGAEVDDFSWEPPTEAETKVLQARRERQDRISRLMGDYLLRGY...,199


In [ ]:
def obtener_candidatos_denovo(limite=2000):
    print("Buscando proteínas diseñadas 'de novo' post-Sept 2021...")
    url_search = "https://search.rcsb.org/rcsbsearch/v2/query"

    query = {
      "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
          {
            "type": "terminal",
            "service": "text",
            "parameters": {
              "attribute": "rcsb_accession_info.initial_release_date",
              "operator": "greater",
              "value": "2021-09-30T00:00:00Z"
            }
          },
          {
            "type": "terminal",
            "service": "text",
            "parameters": {
              "attribute": "rcsb_entry_info.polymer_entity_count_protein",
              "operator": "equals",
              "value": 1
            }
          },
          {
            "type": "terminal", # BUSCAMOS PROTEÍNAS DISEÑADAS
            "service": "text",
            "parameters": {
              "attribute": "struct.title",
              "operator": "contains_phrase",
              "value": "de novo"
            }
          }
        ]
      },
      "return_type": "polymer_entity",
      "request_options": {"paginate": {"start": 0, "rows": limite}}
    }

    response = requests.post(url_search, json=query)
    resultados = response.json().get('result_set', [])
    return [item['identifier'] for item in resultados]

# Obtener IDs, descargar secuencias y correr el clasificador del paso anterior:
ids_denovo = obtener_candidatos_denovo(2000)
df_denovo = obtener_secuencias(ids_denovo)
nuevas_novel, nuevas_similar = clasificar_secuencias(df_denovo)

df_final_novel = pd.concat([df_novel, nuevas_novel])

Buscando proteínas diseñadas 'de novo' post-Sept 2021...
Descargando las secuencias de los candidatos...
Secuencias finales válidas descargadas: 168
Clasificando 168 secuencias... Esto tomará unos minutos.

--- Clasificación Terminada ---
Proteínas 'Novel' (<30% homología pre-2021): 97
Proteínas 'Similar' (>=30% homología pre-2021): 71


In [ ]:
from Bio import Align
import pandas as pd

def filtrar_redundancia_interna(df, limite_identidad=0.30):
    """
    Filtra un DataFrame de secuencias para asegurar que ninguna pareja
    comparta más del límite de identidad (ej. 30%).
    """
    # Configuramos el alineador global
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'

    indices_aprobados = []

    print(f"Iniciando filtro de redundancia para {len(df)} secuencias...")
    print(f"Buscando secuencias con menos de {limite_identidad*100}% de similitud entre ellas.")

    for i, row in df.iterrows():
        seq_candidata = row['Secuencia']
        es_unica = True

        # Comparamos contra las que ya hemos aprobado
        for j in indices_aprobados:
            seq_aprobada = df.loc[j, 'Secuencia']

            # Hacemos el alineamiento
            alineamientos = aligner.align(seq_candidata, seq_aprobada)
            mejor_alineamiento = alineamientos[0]

            # Calculamos el % de identidad (coincidencias / longitud de la más corta)
            longitud_minima = min(len(seq_candidata), len(seq_aprobada))
            coincidencias = str(mejor_alineamiento).count('|')

            identidad = coincidencias / longitud_minima

            if identidad >= limite_identidad:
                es_unica = False
                break

        if es_unica:
            indices_aprobados.append(i)

    df_filtrado = df.loc[indices_aprobados].reset_index(drop=True)
    print(f"¡Listo! Quedaron {len(df_filtrado)} secuencias completamente únicas.")
    return df_filtrado


# 1. Filtramos las "Similar" y nos quedamos con exactamente 30
df_similar_unicas = filtrar_redundancia_interna(df_similar)
df_final_similar = df_similar_unicas.head(30)

# 2. Filtramos las "Novel" (las 3 originales + las de novo que sacaste) y tomamos 30
df_novel_unicas = filtrar_redundancia_interna(df_final_novel)
df_final_novel = df_novel_unicas.head(30)

Iniciando filtro de redundancia para 1847 secuencias...
Buscando secuencias con menos de 30.0% de similitud entre ellas.
¡Listo! Quedaron 154 secuencias completamente únicas.
Iniciando filtro de redundancia para 320 secuencias...
Buscando secuencias con menos de 30.0% de similitud entre ellas.
¡Listo! Quedaron 35 secuencias completamente únicas.


In [ ]:
def guardar_en_fasta(df, nombre_archivo):
    """
    Toma un DataFrame con columnas 'PDB_ID' y 'Secuencia'
    y lo guarda en formato FASTA.
    """
    # Abrimos el archivo en modo escritura ('w')
    with open(nombre_archivo, 'w') as archivo_fasta:
        for index, row in df.iterrows():
            pdb_id = row['PDB_ID']
            secuencia = row['Secuencia']

            # Escribimos el encabezado (ej. >4XYZ)
            archivo_fasta.write(f">{pdb_id}\n")
            # Escribimos la secuencia de aminoácidos
            archivo_fasta.write(f"{secuencia}\n")

    print(f"¡Éxito! Archivo '{nombre_archivo}' guardado con {len(df)} secuencias.")

# --- Ejecución ---

# 1. Guardar las secuencias Similares
guardar_en_fasta(df_similar_unicas.head(30), "similares_WT.fasta")

# 2. Guardar las secuencias Novel
guardar_en_fasta(df_novel_unicas.head(30), "novel_WT.fasta")

¡Éxito! Archivo 'similares_WT.fasta' guardado con 30 secuencias.
¡Éxito! Archivo 'novel_WT.fasta' guardado con 30 secuencias.
